# Notebook 4: Modelos generativos

## Pre-requisitos

### Instalar paquetes

Si la práctica requiere algún paquete de Python, habrá que incluir una celda en la que se instalen. Si usamos un paquete que se ha utilizado en prácticas anteriores, podríamos dar por supuesto que está instalado pero no cuesta nada satisfacer todas las dependencias en la propia práctica para reducir las dependencias entre ellas.

In [1]:
# instalación TensorFlow
# !pip3 install tensorflow
import tensorflow as tf

# Hacemos los imports que sean necesarios
import numpy as np

# Modelos generativos sobre MNIST

Lo primero que tenemos que hacer es cargar el dataset.

In [2]:
labeled_data = 0.01 # Vamos a usar el etiquetado de sólo el 1% de los datos
np.random.seed(42)

(x_train, y_train), (x_test, y_test), = tf.keras.datasets.mnist.load_data()

indexes = np.arange(len(x_train))
np.random.shuffle(indexes)
ntrain_data = int(labeled_data*len(x_train))
unlabeled_train = x_train[indexes[ntrain_data:]]
x_train = x_train[indexes[:ntrain_data]]
y_train = y_train[indexes[:ntrain_data]]

In [3]:
# TODO: Haz el preprocesado que necesites aquí (si lo necesitas)

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Pasamos de dimensión 3 (n_samples, 28, 28) a dim 2 (n_samples, 748)
x_train = x_train.reshape(len(x_train), -1)
unlabeled_train = unlabeled_train.reshape(len(unlabeled_train), -1)
x_test = x_test.reshape(len(x_test), -1)

# convertimos de uint8 a float32
x_train = x_train.astype("float32")
unlabeled_train = unlabeled_train.astype("float32")
x_test = x_test.astype("float32")

# normalizamos y aplicamos PCA
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
unlabeled_train_scaled = scaler.transform(unlabeled_train)
x_test_scaled = scaler.transform(x_test)

pca = PCA(n_components=50, random_state=42)
x_train_pca = pca.fit_transform(x_train_scaled)
unlabeled_train_pca = pca.transform(unlabeled_train_scaled)
x_test_pca = pca.transform(x_test_scaled)

## Modelo generativo

Vamos a crear nuestro propio modelo generativo. En clase de teoría has visto muchas versiones distintas:

1. Mezcla de distribuciones de Gaussianas (GMM)
1. Mezcla de distribuciones multinomiales (Naive Bayes)
1. Modelos de Markov ocultos (HMM)

Tal y como se os apunta en teoría, los modelos generativos abordan un problema más general, y aprenden realmente cómo se estructuran y distribuyen los datos de entrada. 

En nuestro caso, vamos a distribuír los datos de entrada mediante el uso de **Autoencoders**. 

# Autoencoders

El autoencoder es un tipo de red que se utiliza para aprender codificaciones eficientes de datos sin etiquetar (lo que se conoce como aprendizaje no supervisado). Es una red que tiene el mismo tamaño en la entrada como en la salida, puesto que el objetivo de la red es reconstruír la entrada con la menor pérdida posible.

Si lo que hacemos es reconstruír la entrada, ¿qué sentido tiene el usar la red? Habitualmente, **la red consta, a su mitad, de una capa con menos elementos que los datos de entrada**. Por tanto, al reconstruír los datos de la entrada a la salida, en esa capa tendremos una versión *comprimida* de la entrada, que contendrá la mayor parte de su información.

Por tanto, podemos dividir un autoencoder en 3 secciones diferentes, tal y como se ve en la siguiente figura:

![](https://drive.google.com/uc?export=view&id=1yxkKZV0J0YplQAGPGJxQ2Z80Ad6L94eu)

1. **Encoder:** es la parte inicial de la red, encargada de comprimir los datos de la entrada.
1. **Code:** es la salida del encoder, contiene la versión *comprimida* de los datos de entrada.
1. **Decoder:** se encarga de, partiendo de la salida del *Encoder*, reconstruír la red.

## Crea tu propio Autoencoder

El diseño del autoencoder es libre (capas densas, convolucionales, ...), puedes crearlo como quieras. **El único requisito es que tiene que mantener los nombres (y parámetros) de las funciones descritas abajo.**

In [ ]:
# TODO: crea tu propio autoencoder

class MiAutoencoder:

    def __init__(self, input_shape):
        # encoder
        input_layer = tf.keras.layers.Input(shape=(input_shape,))
        x = tf.keras.layers.Dense(256, activation='relu')(input_layer)
        x = tf.keras.layers.Dense(128, activation='relu')(x)
        code = tf.keras.layers.Dense(64, activation='relu')(x)
        # decoder
        x = tf.keras.layers.Dense(128, activation='relu')(code)
        x = tf.keras.layers.Dense(256, activation='relu')(x)
        output_layer = tf.keras.layers.Dense(input_shape, activation='sigmoid')(x)

        self.autoencoder = tf.keras.Model(input_layer, output_layer)
        self.encoder = tf.keras.Model(input_layer, code)
        self.decoder = tf.keras.Model(code, output_layer)

        self.autoencoder.compile(optimizer='adam', loss='mse')

    def fit(self, X, y=None, sample_weight=None):
        self.autoencoder.fit(
            X, X,
            epochs=10,
            batch_size=128,
            shuffle=True,
            sample_weight=sample_weight,
            verbose=1
        )

    def get_encoded_data(self, X):
        return self.encoder.predict(X)

    def __del__(self):
        tf.keras.backend.clear_session()

## Crea tu propio Clasificador

El diseño del clasificador es libre, pero recuerda que tiene que ser simple (máximo dos capas). **El único requisito es que tiene que mantener los nombres (y parámetros) de las funciones descritas abajo.**

In [9]:
# TODO: crea tu propio clasificador

class MiClasificador:

    def __init__(self, input_dim):
        self.model = tf.keras.Sequential([
            tf.keras.layers.Dense(128, activation='relu', input_shape=(input_dim,)),
            tf.keras.layers.Dense(10, activation='softmax')
        ])

        self.model.compile(
            optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    def fit(self, X, y, sample_weight=None):
        self.model.fit(
            X, y,
            epochs=10,
            batch_size=128,
            shuffle=True,
            sample_weight=sample_weight,
            verbose=1
        )

    def predict(self, X):
        probs = self.model.predict(X)
        return np.argmax(probs, axis=1)

    def predict_proba(self, X):
        return self.model.predict(X)

    def score(self, X, y):
        loss, acc = self.model.evaluate(X, y, verbose=0)
        return acc

    def __del__(self):
        tf.keras.backend.clear_session()

### Entrenamiendo del modelo semisupervisado

El entrenamiento del sistema semisupervisado se realiza en dos pasos.

1. Se entrena el autoencoder con todos los datos (etiquetados y sin etiquetar).
1. Se entrena un clasificador simple (una o dos capas), teniendo como entrada la salida del encoder (**code**) de los datos etiquetados.

<font color='red'>NOTA:</font> para entrenar (y predecir) vamos a utilizar los nombres de las funciones que hemos definido en el autoencoder y en el clasificador.

In [6]:
# TODO: implementa el algoritmo semisupervised_training.

def semisupervised_training(autoencoder, classifier, x_train, y_train, unlabeled_data):

    all_data = np.concatenate((x_train, unlabeled_data), axis=0)
    autoencoder.fit(all_data)
    code = autoencoder.get_encoded_data(x_train)
    classifier.fit(code, y_train)

    return autoencoder, classifier

### Entrenamos nuestro modelo

Usa lo hecho anteriormente para entrenar tu clasificador de una manera semi-supervisada.

In [10]:
# Crea tu autoencoder y tu clasificador

autoencoder = MiAutoencoder(input_shape=x_train.shape[1])
classifier = MiClasificador(input_dim=64)  # tamaño del code

c:\Users\Usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
# TODO: Entrena tu modelo

autoencoder, classifier = semisupervised_training(
    autoencoder,
    classifier,
    x_train,
    y_train,
    unlabeled_train
)

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 7218.9839
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 7218.6880
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 7218.6914
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 7218.6885
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 7218.6895
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 7218.6880
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 7218.6914
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 7218.6865
Epoch 9/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 7218.6841
Epoch 10/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 7218.6938
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0950 - loss: 249.3708  
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.1200 - loss: 138.4613 
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.1467 - loss: 63.6610 
Epoc

In [12]:
# TODO: Obtén la precisión sobre el conjunto de test
pred_data = autoencoder.get_encoded_data(x_test)
print('Test accuracy :', classifier.score(pred_data, y_test))

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 646us/step
Test accuracy : 0.21299999952316284


## Mejorando el código

nuestro modelo actual requiere de dos pasos para entrenarse, pero podría realizarse en un único paso si **creamos un modelo con las dos salidas (autoencoder y clasificador)**. 

Para ello, hay que tener en cuenta que, en los datos sin etiquetar, su contribución al clasificador debería ser nula.


### TRABAJO: Crea el nuevo modelo y modifica la función semisupervised_training para tener en cuenta todos los puntos mencionados anteriormente

In [22]:
# TODO: crea el nuevo modelo

# TODO: crea tu propio clasificador

class MiClasificadorSemisupervisado:

    def __init__(self, input_shape):
        inputs = tf.keras.layers.Input(shape=(input_shape,))
        # encoder
        x = tf.keras.layers.Dense(256, activation='relu')(inputs)
        code = tf.keras.layers.Dense(64, activation='relu')(x)
        # decoder
        x_decoder = tf.keras.layers.Dense(256, activation='relu')(code)
        decoding = tf.keras.layers.Dense(input_shape, activation='sigmoid', name="decoding")(x_decoder)
        # clasificador
        x_classifier = tf.keras.layers.Dense(128, activation='relu')(code)
        classification = tf.keras.layers.Dense(10, activation='softmax', name="classification")(x_classifier)

        self.model = tf.keras.Model(inputs=inputs, outputs=[decoding, classification])

        self.model.compile(
            optimizer='adam',
            loss={"decoding": "mse", "classification": "sparse_categorical_crossentropy"},
            metrics={"classification": "accuracy"}
        )

    def fit(self, X, y, unlabeled_data):
        X_all = np.concatenate((X, unlabeled_data), axis=0)

        y_decoding = X_all    # la etiqueta es la propia entrada
        y_classification = np.concatenate((y, np.zeros(len(unlabeled_data))))

        w_decoding = np.ones(len(X_all))
        w_classification = np.concatenate((np.ones(len(X)), np.zeros(len(unlabeled_data))))

        self.model.fit(
            X_all,
            [y_decoding, y_classification],
            sample_weight=[w_decoding, w_classification],
            epochs=20,
            batch_size=256,
            shuffle=True,
            verbose=1
        )

    def predict(self, X):
        _, probs = self.model.predict(X)
        return np.argmax(probs, axis=1)

    def predict_proba(self, X):
        _, probs = self.model.predict(X)
        return probs

    def score(self, X, y):
        preds = self.predict(X)
        return np.mean(preds == y)

    def __del__(self):
        tf.keras.backend.clear_session()

In [23]:
# TODO: reescribe la función semisupervised_training para incorporar las mejoras mencionadas anteriormente

def semisupervised_training_v2(model, x_train, y_train, unlabeled_data):
    model.fit(x_train, y_train, unlabeled_data)
    return model

In [24]:
# TODO: Crea y entrena tu clasificador

model = MiClasificadorSemisupervisado(input_shape=x_train.shape[1])

model = semisupervised_training_v2(
    model,
    x_train,
    y_train,
    unlabeled_train
)

print(model.score(x_test, y_test))

Epoch 1/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - classification_accuracy: 0.1132 - classification_loss: 0.2316 - decoding_loss: 7218.9619 - loss: 7219.2085
Epoch 2/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - classification_accuracy: 0.1035 - classification_loss: 0.0529 - decoding_loss: 7217.3477 - loss: 7217.8281
Epoch 3/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - classification_accuracy: 0.1021 - classification_loss: 0.0294 - decoding_loss: 7217.6709 - loss: 7217.7583
Epoch 4/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - classification_accuracy: 0.1167 - classification_loss: 0.0161 - decoding_loss: 7217.0229 - loss: 7217.5181
Epoch 5/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - classification_accuracy: 0.1026 - classification_loss: 0.0203 - decoding_loss: 7216.9224 - loss: 7217.4526
Epoch 6/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - classification_accuracy: 0.1000 - classification_loss: 0.0137 - decoding_loss: 7216.4180 - loss: 7217.4287
Epoch 7/20
235/235 ━━━━━━━━━━━━━━━

In [25]:
# TODO: Obtén la precisión sobre el conjunto de test
print('Test accuracy :', model.score(x_test, y_test))

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 787us/step
Test accuracy : 0.861


# Hay vida más allá del autoencoder

¿Has probado a utilizar otro método distinto del autoencoder para obtener una respresentación similar a la salida del encoder? La idea es la siguiente:

1. Define un modelo $model$ convolucional similar al encoder de un autoencoder (la entrada es el tamaño de la imagen, la salida el vector de representación)
1. Define una capa de salida $cluster$ que, partiendo de la salida de model, nos devuelva una salida con el mismo número de clases que el dataset a utilizar (la entrada es el vector de representación), usando softmax como activación de salida
1. Para cada batch de entrenamiento $X$:  # Usa un batch alto, mínimo 128
  1. Modifica las imágenes de entrada con [data_augmentation](https://www.tensorflow.org/tutorials/images/data_augmentation?hl=es-419), llámala $augX_1$.
  1. Modifica otra vez las imágenes de entrada con [data_augmentation_2](https://www.tensorflow.org/tutorials/images/data_augmentation?hl=es-419), llámala $augX_2$.
  1. $augX_{1comp} \leftarrow model(augX_1)$
  1. $augX_{2comp} \leftarrow model(augX_2)$
  1. $cX_{1comp} \leftarrow cluster(augX_{1comp})$
  1. $cX_{2comp} \leftarrow cluster(augX_{2comp})$
  1. $M \leftarrow augX_{1comp} ~ augX_{2comp}^T$
  1. $loss_C \leftarrow cX_{1comp}(1 - cX_{1comp}) + cX_{2comp}(1 - cX_{2comp})$ # Puede que tengas que crear tu [propia función de coste](https://keras.io/api/losses/#creating-custom-losses)
  1. $loss_M \leftarrow crossentropy(I, softmax(M/\tau, axis=1)))$ # Puede que tengas que crear tu [propia función de coste](https://keras.io/api/losses/#creating-custom-losses)
    1. $\tau$ es un hiperparámetro que se suele definir a 5.0
  1. $loss \leftarrow loss_M + \lambda~loss_C$
    1. $\lambda$ es un hiperparámetro (puedes probar con 0.5)


In [5]:
# Escribe aquí la solución. Crea tantos bloques de código como necesites. Puedes utilizar la siguiente red para generar distorsiones

data_augmentation = tf.keras.models.Sequential(
    [
        # tf.keras.layers.RandomFlip("horizontal"),  # Puede ser util en otros casos
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomTranslation(0.15, 0.15),
        tf.keras.layers.RandomZoom(.15),
    ]
)

data_augmentation_2 = tf.keras.models.Sequential(
    [
        # tf.keras.layers.RandomFlip("horizontal"),  # Puede ser util en otros casos
        tf.keras.layers.RandomTranslation(0.15, 0.15),
        tf.keras.layers.Resizing(40, 40), # para CIFAR, para MNIST usar 40 en lugar de 48
        tf.keras.layers.RandomCrop(28, 28), # para CIFAR, para MNIST usar 28 en lugar de 32
    ]
)


In [ ]:
class ModeloAug:

    def __init__(self, tau=5.0, lamda=0.5):
        self.tau = tau
        self.lamda = lamda
        self.aug = data_augmentation
        self.aug2 = data_augmentation_2

        inputs = tf.keras.layers.Input(shape=(28,28,1))
        x = tf.keras.layers.Conv2D(32,3,activation='relu')(inputs)
        x = tf.keras.layers.MaxPooling2D()(x)
        x = tf.keras.layers.Conv2D(64,3,activation='relu')(x)
        x = tf.keras.layers.MaxPooling2D()(x)
        x = tf.keras.layers.Flatten()(x)
        code = tf.keras.layers.Dense(128, activation='relu')(x)
        cluster = tf.keras.layers.Dense(10, activation='softmax')(code)

        self.encoder = tf.keras.Model(inputs, code)
        self.cluster_model = tf.keras.Model(inputs, cluster)
        self.optimizer = tf.keras.optimizers.Adam()

    def loss_C(self, clustered1, clustered2):
        return tf.reduce_mean(clustered1*(1-clustered1) + clustered2*(1-clustered2))   # reduce_mean pasa de matriz a escalar

    def loss_M(self, encoded1, encoded2):
        M = tf.matmul(encoded1, encoded2, transpose_b=True)
        identity_matrix = tf.eye((tf.shape(encoded1)[0]))  # matriz identidad
        return tf.reduce_mean(
            tf.keras.losses.categorical_crossentropy(identity_matrix, (tf.nn.softmax(M/self.tau, axis=1)))
        )

    def fit(self, X, epochs=10, batch_size=128):
        dataset = tf.data.Dataset.from_tensor_slices(X).batch(batch_size)
        for epoch in range(epochs):
            for batch in dataset:
                batch = tf.reshape(batch, (-1, 28, 28, 1))
                with tf.GradientTape() as tape:
                    augX1 = self.aug(batch)
                    augX2 = self.aug2(batch)

                    encoded1 = self.encoder(augX1)
                    encoded2 = self.encoder(augX2)
                    
                    clustered1 = self.cluster_model(augX1)
                    clustered2 = self.cluster_model(augX2)

                    lossC = self.loss_C(clustered1, clustered2)
                    lossM = self.loss_M(encoded1, encoded2)

                    loss = lossM + self.lamda*lossC

                vars = self.encoder.trainable_variables + self.cluster_model.trainable_variables    # cogemos todos los pesos
                grads = tape.gradient(loss, vars)     # calculamos los gradientes del loss respecto a cada peso
                self.optimizer.apply_gradients(zip(grads, vars))   # actualizamos pesos
            print("epoch:", epoch, "loss:", float(loss))

    def get_representation(self, X):
        return self.encoder.predict(X)

    def predict_clusters(self, X):
        probs = self.cluster_model.predict(X)
        return np.argmax(probs, axis=1)

    def __del__(self):
        tf.keras.backend.clear_session()

In [11]:
model = ModeloAug()
model.fit(x_train, epochs=10, batch_size=128)
clusters = model.predict_clusters(x_test)

ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input [[[[0.]
   [0.]
   [0.]
   ...
   [0.]
   [0.]
   [0.]]

  [[0.]
   [0.]
   [0.]
   ...
   [0.]
   [0.]
   [0.]]

  [[0.]
   [0.]
   [0.]
   ...
   [0.]
   [0.]
   [0.]]

  ...

  [[0.]
   [0.]
   [0.]
   ...
   [0.]
   [0.]
   [0.]]

  [[0.]
   [0.]
   [0.]
   ...
   [0.]
   [0.]
   [0.]]

  [[0.]
   [0.]
   [0.]
   ...
   [0.]
   [0.]
   [0.]]]


 [[[0.]
   [0.]
   [0.]
   ...
   [0.]
   [0.]
   [0.]]

  [[0.]
   [0.]
   [0.]
   ...
   [0.]
   [0.]
   [0.]]

  [[0.]
   [0.]
   [0.]
   ...
   [0.]
   [0.]
   [0.]]

  ...

  [[0.]
   [0.]
   [0.]
   ...
   [0.]
   [0.]
   [0.]]

  [[0.]
   [0.]
   [0.]
   ...
   [0.]
   [0.]
   [0.]]

  [[0.]
   [0.]
   [0.]
   ...
   [0.]
   [0.]
   [0.]]]


 [[[0.]
   [0.]
   [0.]
   ...
   [0.]
   [0.]
   [0.]]

  [[0.]
   [0.]
   [0.]
   ...
   [0.]
   [0.]
   [0.]]

  [[0.]
   [0.]
   [0.]
   ...
   [0.]
   [0.]
   [0.]]

  ...

  [[0.]
   [0.]
   

# Trabajo extra

¿Has probado a hacer el autoencoder totalmente convolucional? Para el *decoder* puedes usar las funciones [UpSampling2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/UpSampling2D) o [Conv2DTranspose](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2DTranspose).